In [ ]:
import subprocess
for pkg in ["albumentations", "tqdm", "scikit-learn"]:
    subprocess.run(["pip", "install", "-q", pkg])

import os, cv2, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2

print("✓ Imports done")


In [ ]:
def setup_google_drive():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    drive_base = '/content/drive/MyDrive'
    local_dir  = '/content/ham10000'
    img_dir    = os.path.join(local_dir, 'images')
    os.makedirs(img_dir, exist_ok=True)

    part1   = f'{drive_base}/HAM10000_images_part_1'
    part2   = f'{drive_base}/HAM10000_images_part_2'
    csv_src = f'{drive_base}/HAM10000_metadata.csv'

    assert os.path.isfile(csv_src), f"❌ CSV not found: {csv_src}"
    os.system(f"cp '{csv_src}' {local_dir}/HAM10000_metadata.csv")
    print("✓ Metadata CSV copied")

    for folder, label in [(part1, 'part_1'), (part2, 'part_2')]:
        assert os.path.isdir(folder), f"❌ Folder not found: {folder}"
        print(f"  Copying {label}...")
        os.system(f"cp '{folder}'/*.jpg  {img_dir}/ 2>/dev/null")
        os.system(f"cp '{folder}'/*.jpeg {img_dir}/ 2>/dev/null")

    n = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg'))])
    print(f"✓ Total images copied: {n}")
    assert n >= 100, f"❌ Only {n} images found"

    return f'{local_dir}/HAM10000_metadata.csv', img_dir


In [ ]:
class UVFluorescenceTransform:
    """
    Converts visible dermoscopy image to simulated UV fluorescence.
    Based on Wood's lamp physics (365nm wavelength).
    """
    def __init__(self, lesion_type=None):
        self.lesion_type = lesion_type

    def __call__(self, image: np.ndarray) -> np.ndarray:
        img_f = image.astype(np.float32) / 255.0
        lab   = cv2.cvtColor((img_f * 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
        l_ch  = lab[:,:,0].astype(np.float32) / 255.0
        melanin = 1.0 - l_ch
        fluor   = np.clip(1.0 - melanin * 0.7, 0.2, 1.0)

        r, g, b = img_f[:,:,0], img_f[:,:,1], img_f[:,:,2]
        uv_r = np.clip(r * 0.50, 0, 1)
        uv_g = np.clip(g * 0.90 + b * 0.30, 0, 1)
        uv_b = np.clip(b * 1.40 + g * 0.20, 0, 1)
        uv_r *= fluor; uv_g *= fluor; uv_b *= fluor

        if self.lesion_type:
            uv_r, uv_g, uv_b = self._lesion_shift(uv_r, uv_g, uv_b, l_ch)

        yellow = np.array([1.00, 1.05, 0.65])
        uv = np.stack([uv_r, uv_g, uv_b], axis=-1) * yellow
        uv = np.clip(uv + np.random.normal(0, 0.02, uv.shape), 0, 1) * 0.85
        return (uv * 255).astype(np.uint8)

    def _lesion_shift(self, r, g, b, l):
        h, w   = r.shape
        y, x   = np.ogrid[:h, :w]
        mask   = ((x - w/2)**2 + (y - h/2)**2) < (min(h,w)/3)**2
        t      = self.lesion_type

        if t == 'akiec':
            r[mask] = np.clip(r[mask]*1.3+0.2, 0,1)
            g[mask] = np.clip(g[mask]*0.8,     0,1)
            b[mask] = np.clip(b[mask]*0.7,     0,1)
        elif t in ('bkl','df'):
            r[mask] = np.clip(r[mask]*0.9,     0,1)
            g[mask] = np.clip(g[mask]*1.4+0.1, 0,1)
            b[mask] = np.clip(b[mask]*0.8,     0,1)
        elif t == 'mel':
            r[mask] = np.clip(r[mask]*0.6, 0,1)
            g[mask] = np.clip(g[mask]*0.6, 0,1)
            b[mask] = np.clip(b[mask]*0.7, 0,1)
        elif t == 'vasc':
            r[mask] = np.clip(r[mask]*0.5, 0,1)
            g[mask] = np.clip(g[mask]*0.5, 0,1)
            b[mask] = np.clip(b[mask]*0.6, 0,1)
        elif t == 'bcc':
            kernel = np.ones((5,5), np.uint8)
            edge   = mask & ~cv2.erode(mask.astype(np.uint8), kernel).astype(bool)
            r[edge] = np.clip(r[edge]*0.7,     0,1)
            g[edge] = np.clip(g[edge]*0.9,     0,1)
            b[edge] = np.clip(b[edge]*1.3+0.2, 0,1)
        return r, g, b

In [ ]:
CLASS_TO_IDX = {'nv':0,'mel':1,'bkl':2,'bcc':3,'akiec':4,'vasc':5,'df':6}
IDX_TO_CLASS = {v:k for k,v in CLASS_TO_IDX.items()}
CLASS_NAMES  = ['Melanocytic Nevi','Melanoma','Benign Keratosis',
                'Basal Cell Carcinoma','Actinic Keratoses',
                'Vascular Lesions','Dermatofibroma']

def get_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.Resize(224, 224),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1,
                               rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
            A.HueSaturationValue(10, 15, 10, p=0.3),
            A.GaussNoise(p=0.2),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    return A.Compose([
        A.Resize(224, 224),
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ToTensorV2()
    ])


class HAM10000Dataset(Dataset):
    def __init__(self, df, img_dir, mode='train'):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.transform = get_transforms(mode)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row          = self.df.iloc[idx]
        lesion_type  = row['dx']
        label        = CLASS_TO_IDX[lesion_type]

        # Load visible image
        path = self.img_dir / f"{row['image_id']}.jpg"
        if not path.exists():
            path = self.img_dir / f"{row['image_id']}.jpeg"

        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

        # Generate synthetic UV pair
        uv_img = UVFluorescenceTransform(lesion_type=lesion_type)(img)

        vis_t = self.transform(image=img)['image']
        uv_t  = self.transform(image=uv_img)['image']

        return {
            'visible':    vis_t,
            'uv':         uv_t,
            'label':      torch.tensor(label, dtype=torch.long),
            'label_name': lesion_type
        }


def build_dataloaders(csv_path, img_dir, batch_size=32, val_split=0.15):
    df = pd.read_csv(csv_path)

    print("\nClass distribution:")
    print(df['dx'].value_counts().to_string())

    # Stratified train/val split
    train_df, val_df = train_test_split(
        df, test_size=val_split, stratify=df['dx'], random_state=42
    )
    print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")

    # Weighted sampler — fixes class imbalance (nv dominates at ~67%)
    class_counts  = train_df['dx'].value_counts().to_dict()
    sample_weights = [1.0 / class_counts[dx] for dx in train_df['dx']]
    sampler = WeightedRandomSampler(
        weights     = sample_weights,
        num_samples = len(train_df),
        replacement = True
    )

    train_ds = HAM10000Dataset(train_df, img_dir, mode='train')
    val_ds   = HAM10000Dataset(val_df,   img_dir, mode='val')

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=sampler, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader


In [ ]:
class DualStreamResNet18(nn.Module):
    """
    Two ResNet-18 backbones (visible + UV) with cross-attention fusion.
    """
    def __init__(self, num_classes=7, pretrained=True):
        super().__init__()

        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None

        # Separate backbones so each stream learns modality-specific features
        vis_net = resnet18(weights=weights)
        uv_net  = resnet18(weights=weights)

        # Remove final FC layer — use as feature extractor (512-dim output)
        self.visible_backbone = nn.Sequential(*list(vis_net.children())[:-1])
        self.uv_backbone      = nn.Sequential(*list(uv_net.children())[:-1])

        feat_dim = 512  # ResNet-18 output dim

        # Cross-attention: each stream attends to the other
        self.vis_attention = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.Sigmoid())
        self.uv_attention  = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.Sigmoid())

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

        # Embedding head (used ONLY for prototypical extension)
        self.embedding_head = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def extract_features(self, visible, uv):
        """Shared feature extraction + fusion used by both classifier and embedder."""
        vis_f = self.visible_backbone(visible).flatten(1)  # (B, 512)
        uv_f  = self.uv_backbone(uv).flatten(1)            # (B, 512)

        # Cross-attention fusion
        vis_att = self.vis_attention(uv_f)   # UV tells visible what to focus on
        uv_att  = self.uv_attention(vis_f)   # Visible tells UV what to focus on

        vis_attended = vis_f * vis_att
        uv_attended  = uv_f  * uv_att

        fused = vis_attended + uv_attended   # (B, 512)
        return fused

    def forward(self, visible, uv):
        """Standard forward pass for classification training."""
        fused = self.extract_features(visible, uv)
        return self.classifier(fused)

    def get_embedding(self, visible, uv):
        """
        Returns L2-normalized embedding vector.
        Used ONLY by the optional prototypical extension.
        """
        fused = self.extract_features(visible, uv)
        emb   = self.embedding_head(fused)
        return F.normalize(emb, p=2, dim=1)


In [ ]:
def train(model, train_loader, val_loader,
          epochs=25, lr=1e-4, device='cuda', save_dir='/content'):

    model = model.to(device)

    # Two-phase LR: backbone gets lower LR than classifier head
    backbone_params   = list(model.visible_backbone.parameters()) + \
                        list(model.uv_backbone.parameters())
    head_params       = list(model.vis_attention.parameters()) + \
                        list(model.uv_attention.parameters()) + \
                        list(model.classifier.parameters()) + \
                        list(model.embedding_head.parameters())

    optimizer = torch.optim.Adam([
        {'params': backbone_params, 'lr': lr * 0.1},   # backbone: 1e-5
        {'params': head_params,     'lr': lr}           # head:     1e-4
    ], weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

    print(f"\n{'='*60}")
    print(f"  Training: {epochs} epochs | LR={lr} | Device={device}")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{epochs} [train]', leave=False)
        for batch in pbar:
            vis   = batch['visible'].to(device)
            uv    = batch['uv'].to(device)
            lbl   = batch['label'].to(device)

            optimizer.zero_grad()
            out  = model(vis, uv)
            loss = criterion(out, lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            t_loss    += loss.item() * vis.size(0)
            t_correct += (out.argmax(1) == lbl).sum().item()
            t_total   += vis.size(0)
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        t_loss /= t_total
        t_acc   = t_correct / t_total

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0

        with torch.no_grad():
            for batch in val_loader:
                vis = batch['visible'].to(device)
                uv  = batch['uv'].to(device)
                lbl = batch['label'].to(device)

                out  = model(vis, uv)
                loss = criterion(out, lbl)

                v_loss    += loss.item() * vis.size(0)
                v_correct += (out.argmax(1) == lbl).sum().item()
                v_total   += vis.size(0)

        v_loss /= v_total
        v_acc   = v_correct / v_total

        scheduler.step()

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        flag = ''
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), f'{save_dir}/best_model.pth')
            flag = '  ← best'

        print(f"  Epoch {epoch:02d}/{epochs} | "
              f"Train Loss: {t_loss:.4f}  Acc: {t_acc:.4f} | "
              f"Val Loss: {v_loss:.4f}  Acc: {v_acc:.4f}{flag}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history['train_acc'],  label='Train')
    axes[1].plot(history['val_acc'],    label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/training_curves.png', dpi=150)
    plt.show()

    print(f"\n✓ Best Val Accuracy: {best_val_acc:.4f}")
    return model, history


def evaluate(model, val_loader, device='cuda', save_dir='/content'):
    """Full evaluation with per-class report and confusion matrix."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Evaluating'):
            vis = batch['visible'].to(device)
            uv  = batch['uv'].to(device)
            lbl = batch['label']

            out   = model(vis, uv)
            preds = out.argmax(1).cpu()

            all_preds.extend(preds.numpy())
            all_labels.extend(lbl.numpy())

    print("\n── Classification Report ──────────────────────────────────")
    print(classification_report(
        all_labels, all_preds, target_names=CLASS_NAMES, digits=3
    ))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(7)); ax.set_yticks(range(7))
    ax.set_xticklabels([c[:10] for c in CLASS_NAMES], rotation=45, ha='right')
    ax.set_yticklabels([c[:10] for c in CLASS_NAMES])
    plt.colorbar(im)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')

    for i in range(7):
        for j in range(7):
            ax.text(j, i, str(cm[i,j]),
                    ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black',
                    fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{save_dir}/confusion_matrix.png', dpi=150)
    plt.show()

In [ ]:
class PrototypicalExtension:
    """
    Optional bolt-on: lets users add new disease classes
    with just 5-10 images. No retraining needed.

    Uses the trained model's embedding head to place
    new classes in vector space alongside existing ones.
    """
    def __init__(self, model, device='cuda'):
        self.model      = model.to(device)
        self.device     = device
        self.prototypes = {}   # {class_name: (128,) tensor}

    def _get_embedding(self, vis_tensor, uv_tensor):
        self.model.eval()
        vis = vis_tensor.unsqueeze(0).to(self.device)
        uv  = uv_tensor.unsqueeze(0).to(self.device)
        with torch.no_grad():
            return self.model.get_embedding(vis, uv).squeeze(0)

    def build_base_prototypes(self, dataset, n_per_class=20):
        """
        Pre-compute prototypes for existing HAM10000 classes
        so new classes can be compared against them.
        """
        print("\nBuilding base prototypes from training data...")
        for cls_name, cls_idx in CLASS_TO_IDX.items():
            rows = dataset.df[dataset.df['dx'] == cls_name].head(n_per_class)
            embs = []
            for idx in rows.index:
                s = dataset[idx]
                embs.append(self._get_embedding(s['visible'], s['uv']))
            self.prototypes[cls_name] = torch.stack(embs).mean(dim=0)
            print(f"  ✓ {cls_name}: {len(embs)} examples")
        print(f"\n✓ {len(self.prototypes)} base prototypes ready")

    def add_new_class(self, class_name, image_pairs):
        """
        Add new disease with 5-10 (visible, uv) tensor pairs.

        Args:
            class_name:  str  e.g. 'Tropical_Fungal_2026'
            image_pairs: list of (visible_tensor, uv_tensor)
        """
        embs = [self._get_embedding(v, u) for v, u in image_pairs]
        self.prototypes[class_name] = torch.stack(embs).mean(dim=0)
        print(f"✓ New class '{class_name}' added "
              f"({len(embs)} examples, {len(self.prototypes)} total classes)")

    def predict(self, visible, uv, top_k=3):
        """
        Classify using nearest prototype.
        Falls back to this ONLY if predicted class is a custom-added one;
        otherwise use model.forward() for standard HAM10000 classes.
        """
        emb         = self._get_embedding(visible, uv)
        sorted_cls  = sorted(self.prototypes.keys())
        proto_t     = torch.stack([self.prototypes[c] for c in sorted_cls])

        dists = torch.cdist(emb.unsqueeze(0), proto_t).squeeze(0)
        probs = F.softmax(-dists, dim=0)

        top_vals, top_idx = torch.topk(probs, min(top_k, len(sorted_cls)))
        return [(sorted_cls[i.item()], v.item()) for i, v in zip(top_idx, top_vals)]

    def save(self, path):
        torch.save({
            'prototypes': self.prototypes,
        }, path)
        print(f"✓ Prototypes saved to {path}")

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.prototypes = ckpt['prototypes']
        print(f"✓ Loaded {len(self.prototypes)} prototypes")


In [ ]:
if __name__ == "__main__":

    SAVE_DIR   = '/content'
    EPOCHS     = 25
    BATCH_SIZE = 32
    LR         = 1e-4

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n🚀 Device: {device}")
    if device.type == 'cuda':
        print(f"   GPU: {torch.cuda.get_device_name(0)}")

    # ── 1. Load data from Google Drive ───────────────────────────────────────
    print("\n" + "="*60)
    print("  STEP 1: Google Drive")
    print("="*60)
    csv_path, img_dir = setup_google_drive()

    # ── 2. Build dataloaders ──────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  STEP 2: Dataset + Dataloaders")
    print("="*60)
    train_loader, val_loader = build_dataloaders(csv_path, img_dir,
                                                  batch_size=BATCH_SIZE)

    # ── 3. Build model ────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  STEP 3: Dual-Stream ResNet-18")
    print("="*60)
    model = DualStreamResNet18(num_classes=7, pretrained=True)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    # ── 4. Train ──────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  STEP 4: Training")
    print("="*60)
    model, history = train(model, train_loader, val_loader,
                           epochs=EPOCHS, lr=LR, device=device,
                           save_dir=SAVE_DIR)

    # ── 5. Load best checkpoint + evaluate ───────────────────────────────────
    print("\n" + "="*60)
    print("  STEP 5: Evaluation")
    print("="*60)
    model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pth'))
    evaluate(model, val_loader, device=device, save_dir=SAVE_DIR)

    # ── 6. Save final model ───────────────────────────────────────────────────
    torch.save(model.state_dict(), f'{SAVE_DIR}/skinly_final.pth')
    try:
        os.system(f"cp {SAVE_DIR}/skinly_final.pth '/content/drive/MyDrive/skinly_final.pth'")
        print("✓ Model backed up to Google Drive")
    except: pass

    # ── 7. [OPTIONAL] Build prototypical extension ────────────────────────────
    print("\n" + "="*60)
    print("  STEP 6 (Optional): Prototypical Extension")
    print("="*60)

    full_dataset = HAM10000Dataset(
        pd.read_csv(csv_path), img_dir, mode='val'
    )

    proto_ext = PrototypicalExtension(model, device=device)
    proto_ext.build_base_prototypes(full_dataset, n_per_class=20)
    proto_ext.save(f'{SAVE_DIR}/skinly_prototypes.pth')

    # Example: how a user would add a new disease (plug in real images here)
    # new_images = [(visible_tensor_1, uv_tensor_1), ...]
    # proto_ext.add_new_class('New_Disease_Name', new_images)

    print("\n" + "="*60)
    print("  🎉 DONE")
    print("="*60)
    print(f"  skinly_final.pth       — main classification model")
    print(f"  skinly_prototypes.pth  — prototypical extension")
    print(f"  training_curves.png    — loss / accuracy plots")
    print(f"  confusion_matrix.png   — per-class breakdown")
    print("="*60)